# Tests and Documentation

## Overview
dbt provides two types of tests and a built-in documentation system.

**Schema tests (generic tests):** Declared in YAML files; dbt compiles them to SQL under the hood.

| Test | What it checks |
|---|---|
| `unique` | No duplicate values in the column |
| `not_null` | No NULL values in the column |
| `accepted_values` | Column only contains values from a defined list |
| `relationships` | Every value exists as a FK in another model's column |

**Singular tests:** Plain `.sql` files in the `tests/` directory. dbt runs them and expects zero rows returned. If any rows come back, the test fails.

**dbt-utils and dbt-expectations:** Community packages that add dozens of additional generic tests:
- `dbt_utils.expression_is_true` — arbitrary SQL boolean expression per row
- `dbt_utils.unique_combination_of_columns` — composite uniqueness check
- `dbt_expectations.expect_column_values_to_be_between` — range checks
- `dbt_expectations.expect_column_proportion_of_unique_values_to_be_between`

**Documentation:**
- `description:` fields in YAML files are rendered in `dbt docs serve`
- `meta:` blocks add owner, SLA, and custom metadata
- `dbt docs generate` produces `manifest.json` and `catalog.json`
- These feed the interactive lineage DAG browser at `localhost:8080`

---

In [1]:
schema_yml = """
# models/marts/finance/_finance.yml
version: 2

models:
  - name: fct_orders
    description: >
      One row per order. Joins orders, payments, and customer data.
      This is the primary fact table for revenue reporting.
    meta:
      owner: "data-eng@company.com"
      sla: "daily by 06:00 EST"

    columns:
      - name: order_id
        description: "Surrogate primary key"
        tests:
          - unique
          - not_null

      - name: customer_id
        description: "FK to dim_customers"
        tests:
          - not_null
          - relationships:
              to: ref('dim_customers')
              field: customer_id

      - name: order_status
        description: "Current order status"
        tests:
          - accepted_values:
              values: [placed, shipped, completed, return_pending, returned]

      - name: total_amount_usd
        description: "Total payment amount in USD"
        tests:
          - not_null
          - dbt_utils.expression_is_true:
              expression: ">= 0"
"""
print("=== Schema YAML: model descriptions and tests ===")
print(schema_yml)

=== Schema YAML: model descriptions and tests ===

# models/marts/finance/_finance.yml
version: 2

models:
  - name: fct_orders
    description: >
      One row per order. Joins orders, payments, and customer data.
      This is the primary fact table for revenue reporting.
    meta:
      owner: "data-eng@company.com"
      sla: "daily by 06:00 EST"

    columns:
      - name: order_id
        description: "Surrogate primary key"
        tests:
          - unique
          - not_null

      - name: customer_id
        description: "FK to dim_customers"
        tests:
          - not_null
          - relationships:
              to: ref('dim_customers')
              field: customer_id

      - name: order_status
        description: "Current order status"
        tests:
          - accepted_values:
              values: [placed, shipped, completed, return_pending, returned]

      - name: total_amount_usd
        description: "Total payment amount in USD"
        tests:
          - no

In [2]:
singular = """
-- tests/assert_no_negative_payments.sql
-- A singular test: this query should return ZERO rows.
-- dbt FAILS the test if any rows are returned.

SELECT
    order_id,
    total_amount_usd,
    'Negative payment amount' AS failure_reason
FROM {{ ref('fct_orders') }}
WHERE total_amount_usd < 0
"""

orphan = """
-- tests/assert_no_orphan_orders.sql
-- Orders must belong to a known customer

SELECT o.order_id
FROM   {{ ref('fct_orders') }}    AS o
LEFT JOIN {{ ref('dim_customers') }} AS c ON o.customer_id = c.customer_id
WHERE  c.customer_id IS NULL
"""

freshness = """
-- tests/assert_recent_orders.sql
-- fct_orders must have been updated within the last 25 hours

SELECT 1
WHERE NOT EXISTS (
    SELECT 1 FROM {{ ref('fct_orders') }}
    WHERE dbt_updated_at >= NOW() - INTERVAL '25 hours'
)
"""

print("=== Singular tests: SQL that must return zero rows ===")
print("--- assert_no_negative_payments.sql ---")
print(singular)
print("--- assert_no_orphan_orders.sql ---")
print(orphan)
print("--- assert_recent_orders.sql ---")
print(freshness)

=== Singular tests: SQL that must return zero rows ===
--- assert_no_negative_payments.sql ---

-- tests/assert_no_negative_payments.sql
-- A singular test: this query should return ZERO rows.
-- dbt FAILS the test if any rows are returned.

SELECT
    order_id,
    total_amount_usd,
    'Negative payment amount' AS failure_reason
FROM {{ ref('fct_orders') }}
WHERE total_amount_usd < 0

--- assert_no_orphan_orders.sql ---

-- tests/assert_no_orphan_orders.sql
-- Orders must belong to a known customer

SELECT o.order_id
FROM   {{ ref('fct_orders') }}    AS o
LEFT JOIN {{ ref('dim_customers') }} AS c ON o.customer_id = c.customer_id
WHERE  c.customer_id IS NULL

--- assert_recent_orders.sql ---

-- tests/assert_recent_orders.sql
-- fct_orders must have been updated within the last 25 hours

SELECT 1
WHERE NOT EXISTS (
    SELECT 1 FROM {{ ref('fct_orders') }}
    WHERE dbt_updated_at >= NOW() - INTERVAL '25 hours'
)



---
## Running tests and interpreting output

In [3]:
print("=== dbt test output examples ===")
output = """
$ dbt test

Running 12 tests...

PASS  unique_fct_orders_order_id                       [100%] in 0.12s
PASS  not_null_fct_orders_order_id                     [100%] in 0.08s
PASS  not_null_fct_orders_customer_id                  [100%] in 0.09s
PASS  relationships_fct_orders_customer_id__dim_customers  [100%] in 0.15s
PASS  accepted_values_fct_orders_order_status          [100%] in 0.11s
FAIL  not_null_fct_orders_total_amount_usd             [100%] in 0.10s
      Got 3 results, expected 0
PASS  assert_no_negative_payments                      [100%] in 0.13s
PASS  assert_no_orphan_orders                          [100%] in 0.14s
  ...

Finished running 12 tests in 1.42s.
FAIL=1  WARN=0  ERROR=0  SKIP=0  PASS=11
"""
print(output)

print("=== dbt test severity levels ===")
severity = """
tests:
  - not_null:
      severity: error   # default: fails the run, blocks downstream
  - dbt_utils.expression_is_true:
      expression: ">= 0"
      severity: warn    # logs warning but does not fail the run
  - unique:
      config:
        store_failures: true   # save failing rows to a table for inspection
"""
print(severity)

=== dbt test output examples ===

$ dbt test

Running 12 tests...

PASS  unique_fct_orders_order_id                       [100%] in 0.12s
PASS  not_null_fct_orders_order_id                     [100%] in 0.08s
PASS  not_null_fct_orders_customer_id                  [100%] in 0.09s
PASS  relationships_fct_orders_customer_id__dim_customers  [100%] in 0.15s
PASS  accepted_values_fct_orders_order_status          [100%] in 0.11s
FAIL  not_null_fct_orders_total_amount_usd             [100%] in 0.10s
      Got 3 results, expected 0
PASS  assert_no_negative_payments                      [100%] in 0.13s
PASS  assert_no_orphan_orders                          [100%] in 0.14s
  ...

Finished running 12 tests in 1.42s.
FAIL=1  WARN=0  ERROR=0  SKIP=0  PASS=11

=== dbt test severity levels ===

tests:
  - not_null:
      severity: error   # default: fails the run, blocks downstream
  - dbt_utils.expression_is_true:
      expression: ">= 0"
      severity: warn    # logs warning but does not fail the r

---
## Common Pitfalls

**1. Only testing staging models and skipping marts**
Staging tests catch source data issues, but mart tests catch transformation bugs. A transformation error in `fct_orders` that produces NULL `total_amount_usd` for 3 orders is caught only if you test `fct_orders`. Test every model that feeds a dashboard or downstream consumer.

**2. Singular tests that can return false positives**
`assert_recent_orders.sql` uses `NOW()` -- which means the test passes in the morning and fails at night if the model hasn't run yet. Time-based singular tests must account for the pipeline SLA. Use the model's own `dbt_updated_at` column rather than wall-clock time.

**3. Using `severity: warn` for tests that should block pipelines**
`severity: warn` lets the pipeline continue even when a test fails. A `not_null` warning on `customer_id` in `fct_orders` means orders with no customer flow silently into dashboards. Reserve `warn` for informational checks; use the default `error` for data integrity constraints.

**4. Not installing dbt-utils before using its tests**
`dbt_utils.expression_is_true` requires `dbt-utils` in `packages.yml` and `dbt deps` to be run. Referencing packages that are not installed produces a compile error at test time, not a warning -- it blocks the entire test run.

**5. Forgetting `store_failures: true` when debugging a failing test**
By default, a failing test logs the row count but does not show you the offending rows. Adding `store_failures: true` to the test config makes dbt write the failing rows to a `dbt_test__audit` schema table so you can inspect them with `SELECT * FROM dbt_test__audit.not_null_fct_orders_total_amount_usd`.

**6. Writing descriptions only for columns, not for models**
Column-level `description:` fields are valuable, but model-level descriptions are what appear in the lineage graph and are the first thing a new analyst reads. Every mart model should have a model-level description explaining what the model represents, its grain (one row per what?), and any important business logic.


---
*sql_methods_library - Samantha McGarrigle*